# Composite Fundamental Screener — Colab Quickstart

Runs the full pipeline end to end on a **50-ticker subset** of the S&P 500 (set `FULL_UNIVERSE = True` below to scale to all ~500 names — expect ~30 min of rate-limited EDGAR ingestion).

Steps: clone → install → PIT ingest → scores/composite/backtest → validation → key charts inline.

In [ ]:
# If running in Colab, clone the repo first (skip locally):
import os
if not os.path.exists('config.py'):
    # Replace with your repo URL, or upload the project folder to Colab.
    !git clone https://github.com/YOUR_USER/fundamental-screener.git
    %cd fundamental-screener
%pip install -q -e ./pit_fundamentals
%pip install -q -r requirements.txt

In [ ]:
FULL_UNIVERSE = False  # flip to True for the full S&P 500

import os
os.environ.setdefault('SEC_USER_AGENT', 'fundamental-screener colab REPLACE_ME@example.com')

import config
if not FULL_UNIVERSE:
    config.MAX_TICKERS = config.SMALL_UNIVERSE_SIZE  # 50 tickers keeps Colab runtime reasonable

In [ ]:
# 1) Point-in-time ingestion from SEC EDGAR (resumable; disk-cached)
import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s')
from screener.universe import get_sp500_constituents
from pit_fundamentals.ingest import run_ingest

tickers = get_sp500_constituents()['ticker'].tolist()
run_ingest(tickers, db_path=str(config.DB_PATH))

In [ ]:
# 2) Scores -> sector-neutral z -> LASSO composite -> decile backtest
from screener.backtest import run_pipeline
out = run_pipeline()
panel, dec_rets, coefs = out['panel'], out['decile_returns'], out['coefs']
panel.tail()

In [ ]:
# 3) Statistical validation: Newey-West + Deflated Sharpe Ratio
from screener.validation import build_summary, rolling_spread
import config as cfg
summary = build_summary(panel, dec_rets)
summary.to_csv(cfg.VALIDATION_SUMMARY_PATH)
roll = rolling_spread(dec_rets['spread'].dropna())
roll.to_parquet(cfg.ROLLING_SPREAD_PATH)
summary.round(3)

In [ ]:
# 4) Key dashboard charts, rendered inline
from dashboard.app import fig_decile_cumret, fig_rolling, fig_sector_heatmap, fig_f_scatter
fig_decile_cumret(dec_rets).show()
fig_rolling(roll).show()
fig_sector_heatmap(panel.reset_index()).show()
fig_f_scatter(panel.reset_index()).show()

The rolling-spread chart is deliberately titled neutrally — whether it shows persistence or decay is an empirical result, reported as-is. See `FINDINGS.md` for the research memo.